# Logistic Regression from Scratch — Portfolio Edition

**Course:** Machine Learning  
**Dataset:** Loan Approval (train / dev / test splits)  
**Task:** Binary classification — predict loan approval (Y/N)

---

## What this notebook demonstrates

| Component | Implementation |
|---|---|
| Optimizer | SGD · Momentum · **Adam** (from scratch) |
| Regularization | L2 with regularization path analysis |
| Stability | Numerically stable sigmoid · BCE from logits · gradient clipping |
| Stopping | Early stopping with weight restore |
| Scheduling | Cosine annealing · step decay |
| Metrics | Accuracy · F1 · Precision · Recall · **ROC-AUC from scratch** |
| Visualization | Convergence curves · ROC · PR curve · confusion matrix · feature importance · probability histogram |
| Benchmark | Full comparison against sklearn's L-BFGS solver |

---
## 1. Imports & Reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time
import warnings
warnings.filterwarnings('ignore')

# Reproducibility — every result in this notebook is deterministic
SEED = 42
np.random.seed(SEED)

# Raise on overflow so divergence is explicit, not silent
np.seterr(over='raise', invalid='raise')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

print('Environment ready. NumPy:', np.__version__)

---
## 2. Data Loading & Preprocessing

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
train = pd.read_excel('train.xlsx')
dev   = pd.read_excel('dev.xlsx')
test  = pd.read_excel('test.xlsx')

for df in [train, dev, test]:
    df.drop(columns=['Loan_ID'], inplace=True)

for df in [train, dev, test]:
    df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})

print('Shapes  train:', train.shape, ' dev:', dev.shape, ' test:', test.shape)
print('\nClass distribution (train):')
vc = train['Loan_Status'].value_counts()
print(f'  Approved (1): {vc[1]}  ({vc[1]/len(train)*100:.1f}%)')
print(f'  Rejected (0): {vc[0]}  ({vc[0]/len(train)*100:.1f}%)')
train.head(3)

In [ ]:
def handle_missing(train, dev, test):
    """Impute using train statistics only — prevents data leakage."""
    tr, dv, te = train.copy(), dev.copy(), test.copy()
    for col in train.columns:
        if train[col].dtype == 'object':
            fill = train[col].mode()[0]
        else:
            fill = train[col].mean()
        tr[col] = tr[col].fillna(fill)
        dv[col] = dv[col].fillna(fill)
        te[col] = te[col].fillna(fill)
    return tr, dv, te


def encode_data(train, dev, test):
    """One-hot encode; align dev/test columns to train schema."""
    tr = pd.get_dummies(train)
    dv = pd.get_dummies(dev).reindex(columns=tr.columns, fill_value=0)
    te = pd.get_dummies(test).reindex(columns=tr.columns, fill_value=0)
    return tr, dv, te


def split_xy(df):
    X = df.drop(columns=['Loan_Status']).values.astype(float)
    y = df['Loan_Status'].values.astype(float)
    return X, y


def normalize_zscore(X_train, X_dev, X_test):
    """
    Z-score normalization using train statistics only.
    Produces zero-mean, unit-variance inputs — correct for gradient descent.
    Min-max was the original approach; it is outlier-sensitive and does not
    zero-center the inputs, slowing convergence.
    """
    mean = X_train.mean(axis=0)
    std  = X_train.std(axis=0)
    std[std == 0] = 1.0            # constant columns: avoid division by zero
    return (X_train-mean)/std, (X_dev-mean)/std, (X_test-mean)/std


# ── Pipeline ──────────────────────────────────────────────────────────────────
train, dev, test       = handle_missing(train, dev, test)
train, dev, test       = encode_data(train, dev, test)
X_train, y_train       = split_xy(train)
X_dev,   y_dev         = split_xy(dev)
X_test,  y_test        = split_xy(test)
X_train, X_dev, X_test = normalize_zscore(X_train, X_dev, X_test)

feature_names = train.drop(columns=['Loan_Status']).columns.tolist()

print(f'X_train: {X_train.shape}  X_dev: {X_dev.shape}  X_test: {X_test.shape}')
print(f'Features: {len(feature_names)}')

---
## 3. Core Utilities — Numerically Stable Implementations

In [ ]:
# ── Sigmoid ───────────────────────────────────────────────────────────────────
def sigmoid(z):
    """
    Numerically stable sigmoid using piecewise evaluation.

    Naive form  1/(1+exp(-z))  overflows for z << 0  (exp(-z) → inf).
    Stable form uses the identity:
      z >= 0:  sigma(z) = 1 / (1 + exp(-z))          ← no overflow
      z <  0:  sigma(z) = exp(z) / (1 + exp(z))      ← exp(z) stays small
    """
    return np.where(
        z >= 0,
        1.0 / (1.0 + np.exp(-np.clip(z, -500, 500))),
        np.exp(np.clip(z, -500, 500)) / (1.0 + np.exp(np.clip(z, -500, 500)))
    )


# ── Binary Cross-Entropy from logits ──────────────────────────────────────────
def bce_from_logits(y, logits):
    """
    Compute BCE directly from pre-sigmoid logits.

    Using log(sigma(z)) = z - softplus(z)  where softplus(z) = log(1+exp(z)).
    Stable softplus: max(z,0) + log(1+exp(-|z|))  — avoids exp overflow.
    This is identical to PyTorch's BCEWithLogitsLoss.
    """
    z = np.clip(logits, -500, 500)
    softplus = np.maximum(z, 0) + np.log(1 + np.exp(-np.abs(z)))
    return np.mean(softplus - y * z)


# ── Gradient clipping ─────────────────────────────────────────────────────────
def clip_gradients(dw, db, max_norm=5.0):
    """
    Rescale gradient vector if its L2 norm exceeds max_norm.
    Prevents exploding gradients without changing gradient direction.
    """
    grad_norm = np.sqrt(np.dot(dw, dw) + db ** 2)
    if grad_norm > max_norm:
        scale = max_norm / grad_norm
        dw, db = dw * scale, db * scale
    return dw, db


# ── Metrics ───────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, y_proba):
    """
    Returns a dict with accuracy, precision, recall, F1, and AUC.
    All computed from NumPy — no sklearn dependency.
    """
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))

    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)
    accuracy  = (tp + tn) / len(y_true)

    # ROC-AUC from scratch via trapezoidal rule
    thresholds = np.sort(np.unique(y_proba))[::-1]
    P, N = y_true.sum(), (1 - y_true).sum()
    fprs, tprs = [0.0], [0.0]
    for t in thresholds:
        pred = (y_proba >= t).astype(int)
        tprs.append(np.sum((pred==1)&(y_true==1)) / P)
        fprs.append(np.sum((pred==1)&(y_true==0)) / N)
    fprs.append(1.0); tprs.append(1.0)
    auc = np.trapz(tprs, fprs)

    return dict(accuracy=accuracy, precision=precision,
                recall=recall, f1=f1, auc=auc,
                confusion=np.array([[tn, fp], [fn, tp]]),
                fprs=np.array(fprs), tprs=np.array(tprs))


# ── Early stopping ────────────────────────────────────────────────────────────
class EarlyStopping:
    """Monitors val loss; restores best weights when patience is exhausted."""
    def __init__(self, patience=15, min_delta=1e-4):
        self.patience  = patience
        self.min_delta = min_delta
        self.best_loss = np.inf
        self.counter   = 0
        self.best_w    = None
        self.best_b    = None
        self.stopped_epoch = None

    def step(self, val_loss, w, b, epoch):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_w    = w.copy()
            self.best_b    = b
            self.counter   = 0
        else:
            self.counter += 1
        if self.counter >= self.patience:
            self.stopped_epoch = epoch
            return True
        return False


# ── LR schedulers ─────────────────────────────────────────────────────────────
def cosine_annealing(epoch, total_epochs, lr_max=0.1, lr_min=1e-4):
    """Smooth cosine decay from lr_max to lr_min over training."""
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * epoch / total_epochs))


print('Utilities defined.')

---
## 4. Logistic Regression Class — Pluggable Optimizers

In [ ]:
class LogisticRegression:
    """
    Logistic Regression from scratch with NumPy.

    Optimizers
    ----------
    'sgd'      : Mini-batch stochastic gradient descent
    'momentum' : SGD with Polyak momentum (beta=0.9)
    'adam'     : Adaptive Moment Estimation (Kingma & Ba, 2014)
    'adam_cos' : Adam with cosine annealing learning rate schedule

    All optimizers support L2 regularization, gradient clipping,
    and early stopping with best-weight restoration.
    """

    def __init__(self, optimizer='adam', lr=0.001, l2=0.0,
                 batch_size=32, epochs=300, patience=20,
                 clip_norm=5.0, seed=42):
        self.optimizer  = optimizer
        self.lr         = lr
        self.l2         = l2
        self.batch_size = batch_size
        self.epochs     = epochs
        self.patience   = patience
        self.clip_norm  = clip_norm
        self.seed       = seed
        self.w          = None
        self.b          = None
        self.history    = {}

    def _init_weights(self, d):
        np.random.seed(self.seed)
        # Xavier initialization: variance = 2/d
        # Better than zeros — avoids identical first-step gradients
        self.w = np.random.randn(d) * np.sqrt(2.0 / d)
        self.b = 0.0

    def _update(self, dw, db, state, t):
        """Single optimizer step. Modifies self.w, self.b in-place."""
        lr = state.get('lr', self.lr)

        if self.optimizer == 'sgd':
            self.w -= lr * dw
            self.b -= lr * db

        elif self.optimizer in ('momentum',):
            beta = 0.9
            state['vw'] = beta * state['vw'] - lr * dw
            state['vb'] = beta * state['vb'] - lr * db
            self.w += state['vw']
            self.b += state['vb']

        elif self.optimizer in ('adam', 'adam_cos'):
            b1, b2, eps = 0.9, 0.999, 1e-8

            # First moment:  exponential moving average of gradients (mean)
            state['mw'] = b1 * state['mw'] + (1 - b1) * dw
            state['mb'] = b1 * state['mb'] + (1 - b1) * db

            # Second moment: EMA of squared gradients (uncentered variance)
            # Tracks per-parameter gradient scale — enables adaptive lr
            state['vw'] = b2 * state['vw'] + (1 - b2) * dw ** 2
            state['vb'] = b2 * state['vb'] + (1 - b2) * db ** 2

            # Bias correction: moments are zero-biased at t=1,2,...
            # Without this, early updates are too small
            mw_hat = state['mw'] / (1 - b1 ** t)
            mb_hat = state['mb'] / (1 - b1 ** t)
            vw_hat = state['vw'] / (1 - b2 ** t)
            vb_hat = state['vb'] / (1 - b2 ** t)

            # Update: effective lr per-parameter = lr / sqrt(v_hat)
            # Parameters with high gradient variance get smaller effective lr
            self.w -= lr * mw_hat / (np.sqrt(vw_hat) + eps)
            self.b -= lr * mb_hat / (np.sqrt(vb_hat) + eps)

    def _init_state(self, d):
        return {'mw': np.zeros(d), 'mb': 0.0,
                'vw': np.zeros(d), 'vb': 0.0, 'lr': self.lr}

    def fit(self, X_train, y_train, X_val=None, y_val=None, verbose=True):
        n, d = X_train.shape
        self._init_weights(d)
        state = self._init_state(d)
        es    = EarlyStopping(patience=self.patience)
        t     = 0  # global step counter (for Adam bias correction)

        history = {
            'train_loss': [], 'val_loss':   [],
            'train_acc':  [], 'val_acc':    [],
            'grad_norm':  [], 'lr_history': []
        }

        for epoch in range(self.epochs):
            # Cosine annealing: update lr in state each epoch
            if self.optimizer == 'adam_cos':
                state['lr'] = cosine_annealing(
                    epoch, self.epochs, lr_max=self.lr, lr_min=self.lr * 0.01)
            history['lr_history'].append(state.get('lr', self.lr))

            # ── Mini-batch loop ──
            # IMPORTANT: shuffle indices only — never reassign X or y
            # Reassigning mutates the caller's arrays and corrupts experiments
            idx = np.random.permutation(n)
            epoch_grad_norms = []

            for i in range(0, n, self.batch_size):
                batch_idx = idx[i:i + self.batch_size]
                Xb = X_train[batch_idx]
                yb = y_train[batch_idx]

                logits = Xb @ self.w + self.b
                yhat   = sigmoid(logits)

                dw = Xb.T @ (yhat - yb) / len(yb) + self.l2 * self.w
                db = np.mean(yhat - yb)

                dw, db = clip_gradients(dw, db, self.clip_norm)
                epoch_grad_norms.append(np.linalg.norm(dw))

                t += 1
                self._update(dw, db, state, t)

            # ── Epoch-level tracking ──
            tr_logits = X_train @ self.w + self.b
            tr_loss   = bce_from_logits(y_train, tr_logits) + \
                        0.5 * self.l2 * np.dot(self.w, self.w)
            tr_acc    = np.mean((sigmoid(tr_logits) >= 0.5) == y_train)

            history['train_loss'].append(tr_loss)
            history['train_acc'].append(tr_acc)
            history['grad_norm'].append(np.mean(epoch_grad_norms))

            if X_val is not None:
                val_logits = X_val @ self.w + self.b
                val_loss   = bce_from_logits(y_val, val_logits)
                val_acc    = np.mean((sigmoid(val_logits) >= 0.5) == y_val)
                history['val_loss'].append(val_loss)
                history['val_acc'].append(val_acc)

                if es.step(val_loss, self.w, self.b, epoch):
                    self.w = es.best_w
                    self.b = es.best_b
                    if verbose:
                        print(f'[{self.optimizer.upper()}] Early stop '
                              f'at epoch {epoch+1} '
                              f'(best val_loss={es.best_loss:.4f})')
                    break

        self.history = history
        return self

    def predict_proba(self, X):
        return sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def score(self, X, y):
        return np.mean(self.predict(X) == y)


print('LogisticRegression class defined.')

---
## 5. Experiment 1 — Optimizer Convergence Comparison

Train SGD, Momentum, Adam, and Adam+cosine-annealing under identical conditions.
This is the primary result of the project.

In [ ]:
optimizer_configs = {
    'SGD':          dict(optimizer='sgd',      lr=0.05),
    'Momentum':     dict(optimizer='momentum', lr=0.05),
    'Adam':         dict(optimizer='adam',     lr=0.001),
    'Adam+Cosine':  dict(optimizer='adam_cos', lr=0.01),
}

COMMON = dict(l2=0.01, batch_size=32, epochs=300, patience=25, seed=SEED)

trained_models = {}
for name, cfg in optimizer_configs.items():
    print(f'Training {name}...', end=' ')
    t0 = time.perf_counter()
    model = LogisticRegression(**{**COMMON, **cfg})
    model.fit(X_train, y_train, X_dev, y_dev, verbose=False)
    elapsed = time.perf_counter() - t0
    val_acc = model.score(X_dev, y_dev)
    print(f'done in {elapsed:.2f}s  |  val_acc={val_acc:.4f}')
    trained_models[name] = model

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=False)
axes = axes.flatten()

colors = {'SGD': '#E24B4A', 'Momentum': '#BA7517', 'Adam': '#1a56db', 'Adam+Cosine': '#0F6E56'}

for ax, (name, model) in zip(axes, trained_models.items()):
    h = model.history
    epochs_ran = len(h['train_loss'])
    ep = range(epochs_ran)
    c  = colors[name]

    ax.plot(ep, h['train_loss'], color=c,   lw=2,   label='Train loss')
    ax.plot(ep, h['val_loss'],   color=c,   lw=2,   label='Val loss',
            linestyle='--', alpha=0.7)

    ax2 = ax.twinx()
    ax2.plot(ep, h['val_acc'], color='gray', lw=1.2,
             linestyle=':', label='Val acc', alpha=0.6)
    ax2.set_ylabel('Val Accuracy', color='gray', fontsize=9)
    ax2.tick_params(axis='y', labelcolor='gray', labelsize=8)
    ax2.set_ylim(0.5, 1.0)

    final_val = h['val_acc'][-1]
    ax.set_title(f'{name}  (val_acc={final_val:.4f})', fontsize=11, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE Loss')
    ax.legend(fontsize=8, loc='upper right')

fig.suptitle('Optimizer Convergence Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('optimizer_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: optimizer_convergence.png')

---
## 6. Experiment 2 — Gradient Norm Decay

‖∇w‖₂ quantifies how far from a stationary point the optimizer is each epoch.
A well-converged model shows this decaying toward 0.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for name, model in trained_models.items():
    norms = model.history['grad_norm']
    ax.semilogy(norms, label=name, color=colors[name], lw=2)

ax.set_xlabel('Epoch')
ax.set_ylabel('‖∇w‖₂  (log scale)')
ax.set_title('Gradient Norm Decay per Optimizer', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('gradient_norm_decay.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: gradient_norm_decay.png')

---
## 7. Full Evaluation — Best Model on Test Set

In [ ]:
# Select best model by validation accuracy
best_name  = max(trained_models, key=lambda k: trained_models[k].score(X_dev, y_dev))
best_model = trained_models[best_name]
print(f'Best optimizer on dev set: {best_name}')

# Compute all metrics on test set
y_proba_test = best_model.predict_proba(X_test)
y_pred_test  = best_model.predict(X_test)
m = compute_metrics(y_test, y_pred_test, y_proba_test)

print(f"\n{'Metric':<12} {'Value':>8}")
print('-' * 22)
for key in ['accuracy', 'precision', 'recall', 'f1', 'auc']:
    print(f"{key:<12} {m[key]:>8.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── (A) ROC Curve from scratch ────────────────────────────────────────────────
ax = axes[0]
ax.plot(m['fprs'], m['tprs'], color='#1a56db', lw=2.5,
        label=f'From Scratch  AUC={m["auc"]:.4f}')

# Overlay sklearn ROC for validation
try:
    from sklearn.metrics import roc_curve, auc as sk_auc
    fpr_sk, tpr_sk, _ = roc_curve(y_test, y_proba_test)
    ax.plot(fpr_sk, tpr_sk, color='#E24B4A', lw=1.5, linestyle='--',
            label=f'sklearn  AUC={sk_auc(fpr_sk,tpr_sk):.4f}')
except ImportError:
    pass

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.4, label='Random baseline')
ax.set_xlabel('False Positive Rate');  ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontweight='bold')
ax.legend(fontsize=9)

# ── (B) Precision-Recall Curve ────────────────────────────────────────────────
ax = axes[1]
thresholds = np.linspace(0.01, 0.99, 200)
precisions, recalls, f1s = [], [], []
for t in thresholds:
    preds = (y_proba_test >= t).astype(int)
    tp = np.sum((preds==1)&(y_test==1))
    fp = np.sum((preds==1)&(y_test==0))
    fn = np.sum((preds==0)&(y_test==1))
    p  = tp / (tp + fp + 1e-9)
    r  = tp / (tp + fn + 1e-9)
    precisions.append(p);  recalls.append(r)
    f1s.append(2*p*r/(p+r+1e-9))

ax.plot(recalls, precisions, color='#0F6E56', lw=2.5)
best_f1_idx = np.argmax(f1s)
ax.scatter(recalls[best_f1_idx], precisions[best_f1_idx],
           color='#E24B4A', s=80, zorder=5,
           label=f'Best F1={f1s[best_f1_idx]:.3f} @ t={thresholds[best_f1_idx]:.2f}')
prevalence = y_test.mean()
ax.axhline(prevalence, color='gray', lw=1, linestyle='--',
           label=f'Majority baseline ({prevalence:.2f})')
ax.set_xlabel('Recall');  ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve', fontweight='bold')
ax.legend(fontsize=9)

# ── (C) Confusion Matrix ──────────────────────────────────────────────────────
ax = axes[2]
cm = m['confusion']  # [[TN, FP], [FN, TP]]
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
labels = ['Rejected (0)', 'Approved (1)']
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i,j]}\n({cm_norm[i,j]:.1%})',
                ha='center', va='center', fontsize=11,
                color='white' if cm_norm[i,j] > 0.5 else 'black')
ax.set_xticks([0,1]); ax.set_xticklabels(labels, rotation=15, fontsize=9)
ax.set_yticks([0,1]); ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (Normalized)', fontweight='bold')

fig.suptitle(f'Test Set Evaluation — {best_name}  '
             f'(Acc={m["accuracy"]:.4f}  F1={m["f1"]:.4f}  AUC={m["auc"]:.4f})',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('evaluation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: evaluation_metrics.png')

---
## 8. Experiment 3 — Decision Threshold Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(thresholds, precisions, label='Precision', color='#1a56db', lw=2)
ax.plot(thresholds, recalls,    label='Recall',    color='#E24B4A', lw=2)
ax.plot(thresholds, f1s,        label='F1-score',  color='#0F6E56', lw=2.5)
ax.axvline(thresholds[best_f1_idx], color='gray', lw=1.2, linestyle='--',
           label=f'Optimal F1 threshold = {thresholds[best_f1_idx]:.2f}')
ax.axvline(0.5, color='black', lw=1, linestyle=':', alpha=0.5, label='Default threshold (0.5)')

ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs Decision Threshold', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: threshold_analysis.png')
print(f'\nNote: For a loan approval system, adjusting the threshold from 0.5 to '
      f'{thresholds[best_f1_idx]:.2f} maximizes F1.')
print('In practice, a lender might prefer a lower threshold (higher recall)'
      ' to minimize missed approvals.')

---
## 9. Experiment 4 — Feature Importance via Weight Magnitude

After Z-score normalization, |w_i| directly reflects each feature's influence on the decision boundary.

In [ ]:
w = best_model.w
importance = np.abs(w)
sorted_idx = np.argsort(importance)[::-1]

# Show top 15 features
top_n = min(15, len(feature_names))
top_idx = sorted_idx[:top_n]

fig, ax = plt.subplots(figsize=(9, 6))
bar_colors = ['#1a56db' if w[i] > 0 else '#E24B4A' for i in top_idx]
bars = ax.barh(
    range(top_n),
    importance[top_idx],
    color=bar_colors, alpha=0.85, edgecolor='white'
)
ax.set_yticks(range(top_n))
ax.set_yticklabels([feature_names[i] for i in top_idx], fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('|Weight|  (Z-score normalized features)')
ax.set_title('Feature Importance by Weight Magnitude', fontweight='bold')

# Legend for sign
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#1a56db', label='Positive → increases approval probability'),
                   Patch(facecolor='#E24B4A', label='Negative → decreases approval probability')]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: feature_importance.png')

---
## 10. Experiment 5 — L2 Regularization Path

As λ increases, weights shrink toward 0 at different rates — revealing which features are robust vs. noise-driven.

In [ ]:
lambdas = np.logspace(-4, 2, 25)   # 1e-4 to 1e2, log-spaced
weight_paths = []   # shape: (n_lambdas, n_features)
dev_f1s      = []

for lam in lambdas:
    m_reg = LogisticRegression(
        optimizer='adam', lr=0.001, l2=lam,
        batch_size=32, epochs=200, patience=15, seed=SEED
    )
    m_reg.fit(X_train, y_train, X_dev, y_dev, verbose=False)
    weight_paths.append(m_reg.w.copy())
    y_pred_dev = m_reg.predict(X_dev)
    y_prob_dev = m_reg.predict_proba(X_dev)
    met = compute_metrics(y_dev, y_pred_dev, y_prob_dev)
    dev_f1s.append(met['f1'])

weight_paths = np.array(weight_paths)   # (n_lambdas, n_features)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Regularization path ───────────────────────────────────────────────────────
ax = axes[0]
# Plot top-5 most variable features across the path
weight_std = weight_paths.std(axis=0)
top5_idx   = np.argsort(weight_std)[::-1][:5]
cmap = plt.cm.tab10
for rank, fi in enumerate(top5_idx):
    ax.semilogx(lambdas, weight_paths[:, fi],
                label=feature_names[fi], lw=2, color=cmap(rank))
ax.axhline(0, color='black', lw=0.8, linestyle='--', alpha=0.4)
ax.set_xlabel('λ  (L2 regularization strength)')
ax.set_ylabel('Weight value')
ax.set_title('Regularization Path (top-5 features)', fontweight='bold')
ax.legend(fontsize=8)

# ── F1 vs Lambda ──────────────────────────────────────────────────────────────
ax = axes[1]
ax.semilogx(lambdas, dev_f1s, color='#1a56db', lw=2.5, marker='o', markersize=4)
best_lam_idx = np.argmax(dev_f1s)
ax.axvline(lambdas[best_lam_idx], color='#E24B4A', lw=1.5, linestyle='--',
           label=f'Best λ={lambdas[best_lam_idx]:.4f}  F1={dev_f1s[best_lam_idx]:.4f}')
ax.set_xlabel('λ  (L2 regularization strength)')
ax.set_ylabel('Dev F1-score')
ax.set_title('Dev F1 vs Regularization Strength', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('regularization_path.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: regularization_path.png')
print(f'Optimal λ = {lambdas[best_lam_idx]:.5f}')

---
## 11. Experiment 6 — Predicted Probability Distribution

In [ ]:
y_proba_test = best_model.predict_proba(X_test)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(y_proba_test[y_test == 0], bins=30, alpha=0.65,
        color='#E24B4A', label='Rejected (true label=0)', density=True)
ax.hist(y_proba_test[y_test == 1], bins=30, alpha=0.65,
        color='#1a56db', label='Approved (true label=1)', density=True)
ax.axvline(0.5, color='black', lw=1.2, linestyle='--', label='Decision threshold (0.5)')
ax.set_xlabel('Predicted Probability of Approval')
ax.set_ylabel('Density')
ax.set_title('Predicted Probability Distribution by True Class', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('probability_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: probability_distribution.png')
print('\nInterpretation: Two well-separated peaks near 0 and 1 indicate a')
print('confident, well-calibrated classifier. Overlap near 0.5 represents')
print('ambiguous cases where the model is uncertain.')

---
## 12. Benchmark — From Scratch vs. sklearn (L-BFGS)

sklearn uses L-BFGS, a second-order optimizer that approximates the Hessian.
Our Adam implementation uses only first-order gradients.
The gap between them quantifies what second-order curvature information buys on this dataset.

In [ ]:
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score
)

best_lam = lambdas[np.argmax(dev_f1s)]

# ── Train sklearn (L-BFGS) ────────────────────────────────────────────────────
sk = SklearnLR(C=1.0/best_lam, max_iter=1000, solver='lbfgs',
               random_state=SEED)
t0 = time.perf_counter()
sk.fit(X_train, y_train)
sk_time = time.perf_counter() - t0

# ── Train our best model (Adam) ───────────────────────────────────────────────
my = LogisticRegression(
    optimizer='adam', lr=0.001, l2=best_lam,
    batch_size=32, epochs=300, patience=25, seed=SEED
)
t0 = time.perf_counter()
my.fit(X_train, y_train, X_dev, y_dev, verbose=True)
my_time = time.perf_counter() - t0

# ── Metrics ───────────────────────────────────────────────────────────────────
def get_metrics(model, X, y, is_sklearn=False):
    if is_sklearn:
        preds = model.predict(X)
        proba = model.predict_proba(X)[:, 1]
    else:
        preds = model.predict(X)
        proba = model.predict_proba(X)
    return {
        'Accuracy':  round(accuracy_score(y, preds), 4),
        'Precision': round(precision_score(y, preds, zero_division=0), 4),
        'Recall':    round(recall_score(y, preds, zero_division=0), 4),
        'F1':        round(f1_score(y, preds, zero_division=0), 4),
        'AUC':       round(roc_auc_score(y, proba), 4),
    }

my_metrics = get_metrics(my, X_test, y_test, is_sklearn=False)
sk_metrics = get_metrics(sk, X_test, y_test, is_sklearn=True)

comparison_df = pd.DataFrame({
    'Metric':             list(my_metrics.keys()),
    'Scratch (Adam)':     list(my_metrics.values()),
    'sklearn (L-BFGS)':   list(sk_metrics.values()),
})
comparison_df['Gap'] = (
    comparison_df['sklearn (L-BFGS)'] - comparison_df['Scratch (Adam)']
).round(4)

print('\n=== Test Set Benchmark ===' )
print(comparison_df.to_string(index=False))
print(f'\nTrain time — Scratch (Adam): {my_time:.3f}s  |  sklearn (L-BFGS): {sk_time:.3f}s')

# Weight alignment: corr > 0.99 means both found the same optimum
corr = np.corrcoef(my.w, sk.coef_[0])[0, 1]
print(f'\nWeight vector correlation (Scratch vs sklearn): {corr:.4f}')
print('(Correlation > 0.99 confirms both optimizers converged to the same solution.)')

In [ ]:
# Visual comparison: bar chart of all metrics
metrics_list = list(my_metrics.keys())
my_vals      = list(my_metrics.values())
sk_vals      = list(sk_metrics.values())

x   = np.arange(len(metrics_list))
w   = 0.35
fig, ax = plt.subplots(figsize=(10, 5))

b1 = ax.bar(x - w/2, my_vals, w, label='Scratch (Adam)',   color='#1a56db', alpha=0.85)
b2 = ax.bar(x + w/2, sk_vals, w, label='sklearn (L-BFGS)', color='#E24B4A', alpha=0.85)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.003,
            f'{bar.get_height():.4f}',
            ha='center', va='bottom', fontsize=8.5)

ax.set_ylim(0.6, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(metrics_list)
ax.set_ylabel('Score')
ax.set_title('From Scratch vs sklearn — Test Set Metrics', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('sklearn_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: sklearn_benchmark.png')

---
## 13. Experiment 7 — Learning Rate Sensitivity (Original Experiment, Fixed)

Reproduces the original LR sweep experiment with the data mutation bug fixed
and divergence detection made explicit.

In [ ]:
learning_rates = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]
lr_results = []

for lr in learning_rates:
    try:
        m_lr = LogisticRegression(
            optimizer='adam', lr=lr, l2=0.01,
            batch_size=32, epochs=200, patience=20, seed=SEED
        )
        m_lr.fit(X_train, y_train, X_dev, y_dev, verbose=False)

        # Explicit divergence check — not a bare except
        if np.isnan(m_lr.w).any() or np.isinf(m_lr.w).any():
            print(f'LR {lr:.0e}: diverged (NaN/Inf weights)')
            lr_results.append((lr, None, None))
            continue

        tr_acc  = m_lr.score(X_train, y_train)
        dev_acc = m_lr.score(X_dev, y_dev)
        lr_results.append((lr, tr_acc, dev_acc))
        print(f'LR {lr:.0e}: train_acc={tr_acc:.4f}  dev_acc={dev_acc:.4f}')

    except FloatingPointError:
        print(f'LR {lr:.0e}: numerical overflow — diverged')
        lr_results.append((lr, None, None))

lr_df = pd.DataFrame(lr_results, columns=['LR', 'Train Acc', 'Dev Acc'])
print('\n', lr_df)

In [ ]:
valid = lr_df.dropna()
fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(valid['LR'], valid['Train Acc'], 'o-', color='#1a56db', lw=2, label='Train Acc')
ax.semilogx(valid['LR'], valid['Dev Acc'],   's--', color='#E24B4A', lw=2, label='Dev Acc')
ax.set_xlabel('Learning Rate (log scale)')
ax.set_ylabel('Accuracy')
ax.set_title('Learning Rate Sensitivity (Adam optimizer)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('lr_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: lr_sensitivity.png')

---
## 14. Summary Table

In [ ]:
from sklearn.metrics import f1_score as sk_f1, roc_auc_score

rows = []
for name, model in trained_models.items():
    yp = model.predict(X_test)
    yprob = model.predict_proba(X_test)
    rows.append({
        'Optimizer':  name,
        'Accuracy':   round(model.score(X_test, y_test), 4),
        'F1':         round(sk_f1(y_test, yp, zero_division=0), 4),
        'AUC':        round(roc_auc_score(y_test, yprob), 4),
        'Epochs run': len(model.history['train_loss']),
    })

# Add sklearn
sk_yp = sk.predict(X_test)
sk_yprob = sk.predict_proba(X_test)[:, 1]
rows.append({
    'Optimizer':  'sklearn (L-BFGS)',
    'Accuracy':   round(accuracy_score(y_test, sk_yp), 4),
    'F1':         round(sk_f1(y_test, sk_yp, zero_division=0), 4),
    'AUC':        round(roc_auc_score(y_test, sk_yprob), 4),
    'Epochs run': 'N/A',
})

summary_df = pd.DataFrame(rows).set_index('Optimizer')
print('=== Final Results Summary ===')
print(summary_df.to_string())
summary_df